# 📈 Clase 2 — Regresión lineal simple
## Unidad: Correlación y modelamiento

**Situación:** El equipo de marketing quiere estimar cuánto gastará un cliente en función del tiempo que pasa navegando en la tienda online. Necesitas construir un modelo que permita hacer esa predicción.

**Preguntas clave:**
- ¿Se puede predecir el gasto a partir del tiempo de navegación?
- ¿Cómo sabemos si el modelo es bueno?

**Objetivos:**
- Entender la estructura del modelo Y = β₀ + β₁X + ε
- Calcular coeficientes con **OLS** (`statsmodels`)
- Interpretar intercepto (β₀) y pendiente (β₁)
- Evaluar con métricas de error: **MAE**, **MSE**, **RMSE**
- Medir ajuste con **R²**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

print('✅ Librerías cargadas')
print(f'statsmodels {sm.__version__} | pandas {pd.__version__}')

---
## PARTE 1 — Estructura del modelo de regresión lineal simple

### 1.1 Ejemplo introductorio de la presentación

In [ ]:
# Datos exactos de la presentación
df_intro = pd.DataFrame({
    'tiempo': [1, 2, 3, 4, 5, 6, 7],
    'gasto':  [20, 23, 25, 30, 32, 34, 38]
})

X_i = sm.add_constant(df_intro['tiempo'])
Y_i = df_intro['gasto']
modelo_intro = sm.OLS(Y_i, X_i).fit()

print(modelo_intro.summary())

In [ ]:
b0 = modelo_intro.params['const']
b1 = modelo_intro.params['tiempo']

print(f'β₀ (Intercepto): {b0:.2f}')
print(f'β₁ (Pendiente):  {b1:.2f}')
print()
print('Interpretación:')
print(f'  Si una persona no navega (X=0), se espera un gasto base de ${b0:.2f}')
print(f'  Por cada minuto adicional de navegación, el gasto aumenta en ${b1:.2f}')
print()
print(f'Ecuación del modelo: ŷ = {b0:.2f} + {b1:.2f} × tiempo')
print()
# Predicción ejemplo
tiempo_nuevo = 5
pred_nueva = b0 + b1 * tiempo_nuevo
print(f'Predicción para tiempo = {tiempo_nuevo} min: ŷ = ${pred_nueva:.2f}')

### 1.2 Cálculo manual de β₁ y β₀ (fórmulas OLS)

In [ ]:
X_vals = df_intro['tiempo'].values
Y_vals = df_intro['gasto'].values
X_mean, Y_mean = X_vals.mean(), Y_vals.mean()

numerador   = np.sum((X_vals - X_mean) * (Y_vals - Y_mean))
denominador = np.sum((X_vals - X_mean) ** 2)

b1_manual = numerador / denominador
b0_manual = Y_mean - b1_manual * X_mean

print('=== Cálculo manual de coeficientes OLS ===')
print(f'X̄ = {X_mean:.2f} | Ȳ = {Y_mean:.2f}')
print(f'Σ(Xi-X̄)(Yi-Ȳ) = {numerador:.2f}')
print(f'Σ(Xi-X̄)²      = {denominador:.2f}')
print(f'β₁ = {numerador:.2f} / {denominador:.2f} = {b1_manual:.4f}')
print(f'β₀ = Ȳ - β₁·X̄ = {Y_mean:.2f} - {b1_manual:.4f}·{X_mean:.2f} = {b0_manual:.4f}')
print()
print(f'Verificación con statsmodels → β₀={b0:.4f} | β₁={b1:.4f} ✅')

In [ ]:
# Visualización del modelo con residuos
pred_i = modelo_intro.predict(X_i)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Regresión Lineal Simple — Ejemplo introductorio', fontweight='bold')

# Scatterplot + recta de regresión
axes[0].scatter(df_intro['tiempo'], df_intro['gasto'],
                color='#2E75B6', s=80, zorder=5, label='Datos reales')
x_line = np.linspace(0.5, 7.5, 100)
axes[0].plot(x_line, b0 + b1 * x_line, color='#ED7D31', linewidth=2.5,
             label=f'ŷ = {b0:.2f} + {b1:.2f}·X')
# Residuos
for xi, yi, pi in zip(df_intro['tiempo'], df_intro['gasto'], pred_i):
    axes[0].plot([xi, xi], [yi, pi], color='#A5A5A5', linestyle='--', linewidth=1)
axes[0].set_title('Recta de regresión + residuos (líneas grises)')
axes[0].set_xlabel('Tiempo de navegación (min)')
axes[0].set_ylabel('Gasto')
axes[0].legend()

# Residuos vs valores predichos
residuos = Y_i - pred_i
axes[1].scatter(pred_i, residuos, color='#70AD47', s=80, alpha=0.8)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Gráfico de residuos (deben distribuirse alrededor de 0)')
axes[1].set_xlabel('Valores predichos (ŷ)')
axes[1].set_ylabel('Residuos (Y - ŷ)')

plt.tight_layout()
plt.show()

### ✏️ Ejercicio 1 — Reflexiona:

In [ ]:
# ✏️ ¿Qué significa un residuo positivo? ¿Y uno negativo?
r_residuo = ""

# ✏️ ¿Qué esperaríamos ver en el gráfico de residuos si el modelo es bueno?
r_buen_modelo = ""

# ✏️ ¿Cuánto gastaría una persona que navega 10 minutos? Calcula con la ecuación:
pred_10 = b0 + b1 * 10
print(f'Predicción para 10 min: ŷ = {b0:.2f} + {b1:.2f}×10 = {pred_10:.2f}')

# ✏️ ¿Tiene sentido interpretar β₀ en este contexto (tiempo=0)?
r_intercepto = ""

print(f'\nResiduos: {r_residuo}')
print(f'Buen modelo: {r_buen_modelo}')
print(f'β₀ con X=0: {r_intercepto}')

---
## PARTE 2 — Métricas de error y R²

### 2.1 MAE, MSE, RMSE — calculados paso a paso

In [ ]:
# Código exacto de la presentación
predicciones = modelo_intro.predict(X_i)
mae  = np.mean(np.abs(Y_i - predicciones))
mse  = np.mean((Y_i - predicciones) ** 2)
rmse = np.sqrt(mse)

print('=== Métricas de error — Ejemplo introductorio ===')
print(f'MAE:   {mae:.4f}  → error medio absoluto en las mismas unidades de Y')
print(f'MSE:   {mse:.4f}  → penaliza errores grandes (en unidades²)')
print(f'RMSE:  {rmse:.4f} → interpretable en las mismas unidades de Y')
print()
# Contexto: comparar RMSE con desv.std de Y
std_y = Y_i.std()
print(f'Desviación estándar de Y: {std_y:.4f}')
print(f'RMSE / std(Y):             {rmse/std_y:.3f}  → si < 0.5, el modelo es razonable')

In [ ]:
# Mostrar el cálculo de cada error individualmente
tabla_errores = pd.DataFrame({
    'X (tiempo)': df_intro['tiempo'].values,
    'Y real':     Y_i.values,
    'ŷ pred':     predicciones.round(2).values,
    'Residuo (Y-ŷ)': (Y_i - predicciones).round(2).values,
    '|Residuo|':  np.abs(Y_i - predicciones).round(2).values,
    'Residuo²':   ((Y_i - predicciones)**2).round(2).values
})

print('=== Tabla de residuos ===')
print(tabla_errores.to_string(index=False))
print()
print(f'Suma |Residuo| / n = {tabla_errores["|Residuo|"].sum():.4f} / {len(Y_i)} = MAE = {mae:.4f}')
print(f'Suma Residuo² / n  = {tabla_errores["Residuo²"].sum():.4f} / {len(Y_i)} = MSE = {mse:.4f}')

### 2.2 Coeficiente de determinación R²

In [ ]:
# Código exacto de la presentación
r2 = modelo_intro.rsquared
print(f'R² = {r2:.4f}')
print(f'Interpretación: el modelo explica el {r2*100:.1f}% de la variabilidad de Y')
print()

# Cálculo manual de R²
SSE = np.sum((Y_i - predicciones) ** 2)          # suma cuadrados del error
SST = np.sum((Y_i - Y_i.mean()) ** 2)            # suma total de cuadrados
r2_manual = 1 - (SSE / SST)

print(f'Cálculo manual: R² = 1 - SSE/SST = 1 - {SSE:.4f}/{SST:.4f} = {r2_manual:.4f}')
print()

# Escala de interpretación
tabla_r2 = pd.DataFrame({
    'Rango R²': ['≥ 0.90', '0.70–0.89', '0.50–0.69', '< 0.50'],
    'Interpretación': ['Ajuste excelente','Ajuste bueno','Ajuste moderado','Ajuste débil']
})
print(tabla_r2.to_string(index=False))

### ✏️ Ejercicio 2 — Completa:

In [ ]:
# ✏️ ¿Cuál es la diferencia entre MAE y RMSE? ¿Cuándo preferirías cada uno?
r_mae_vs_rmse = ""

# ✏️ ¿Qué significa un R² = 0.95? ¿Y un R² = 0.15?
r_r2_alto = ""
r_r2_bajo = ""

# ✏️ Un R² alto, ¿implica que X causa Y? ¿Por qué?
r_causalidad = ""

print(f'MAE vs RMSE: {r_mae_vs_rmse}')
print(f'R²=0.95:     {r_r2_alto}')
print(f'R²=0.15:     {r_r2_bajo}')
print(f'R² y causa:  {r_causalidad}')

---
## PARTE 3 — Actividad guiada: Predicción del gasto

### 3.1 Cargar y explorar `navegacion_clientes.csv`

In [ ]:
# Código exacto de la presentación
df = pd.read_csv('navegacion_clientes.csv')
print(df.head())
print()
print(df.describe().round(2))

In [ ]:
print('=== Info y tipos de dato ===')
print(df.info())
print()
print('=== Valores faltantes ===')
print(df.isnull().sum())
print()
print('=== Valores fuera de rango ===')
print(df[df['tiempo_navegacion'] < 0])

In [ ]:
# Limpieza de los errores intencionales
df_clean = df.dropna(subset=['gasto']).copy()
df_clean = df_clean[df_clean['tiempo_navegacion'] > 0]
df_clean['gasto'] = pd.to_numeric(df_clean['gasto'], errors='coerce')
df_clean = df_clean.dropna()

print(f'Registros originales: {len(df)} | Después de limpieza: {len(df_clean)}')

### 3.2 Exploración visual (antes de ajustar)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Exploración previa — navegacion_clientes.csv', fontweight='bold')

# Scatterplot
sns.scatterplot(data=df_clean, x='tiempo_navegacion', y='gasto',
                ax=axes[0], color='#2E75B6', alpha=0.6, s=40)
axes[0].set_title('Dispersión: tiempo vs gasto')
axes[0].set_xlabel('Tiempo (min)')
axes[0].set_ylabel('Gasto ($)')

# Histograma de gasto
sns.histplot(df_clean['gasto'], bins=15, kde=True, ax=axes[1], color='#BDD7EE')
axes[1].set_title('Distribución del gasto')
axes[1].set_xlabel('Gasto ($)')

# Histograma de tiempo
sns.histplot(df_clean['tiempo_navegacion'], bins=15, kde=True, ax=axes[2], color='#E2EFDA')
axes[2].set_title('Distribución del tiempo de navegación')
axes[2].set_xlabel('Tiempo (min)')

r_prev = df_clean['tiempo_navegacion'].corr(df_clean['gasto'])
axes[0].set_title(f'Dispersión (r = {r_prev:.3f})')

plt.tight_layout()
plt.show()

### 3.3 Ajustar el modelo de regresión lineal

In [ ]:
# Código exacto de la presentación
X = sm.add_constant(df_clean['tiempo_navegacion'])
Y = df_clean['gasto']
modelo = sm.OLS(Y, X).fit()
print(modelo.summary())

In [ ]:
# Extraer e interpretar coeficientes
b0_g = modelo.params['const']
b1_g = modelo.params['tiempo_navegacion']
pval = modelo.pvalues['tiempo_navegacion']

print('=== Coeficientes del modelo ===')
print(f'β₀ (Intercepto): ${b0_g:,.0f}')
print(f'β₁ (Pendiente):  ${b1_g:,.0f} por minuto de navegación')
print(f'p-value β₁:      {pval:.6f} → {"estadísticamente significativo ✅" if pval < 0.05 else "NO significativo ⚠️"}')
print()
print(f'Ecuación: ŷ = ${b0_g:,.0f} + ${b1_g:,.0f} × tiempo_navegacion')
print()
print('Interpretación contextual:')
print(f'  Gasto base estimado (sin navegación): ${b0_g:,.0f}')
print(f'  Por cada minuto adicional de navegación, el gasto aumenta en ${b1_g:,.0f}')

### 3.4 Calcular métricas de error y R²

In [ ]:
# Código exacto de la presentación
pred = modelo.predict(X)
mae  = np.mean(np.abs(Y - pred))
mse  = np.mean((Y - pred) ** 2)
rmse = np.sqrt(mse)
r2   = modelo.rsquared

print('=== Métricas del modelo — Actividad guiada ===')
print(f'MAE:   ${mae:,.0f}  → error promedio absoluto en predicciones')
print(f'MSE:   ${mse:,.0f}  → penaliza errores grandes')
print(f'RMSE:  ${rmse:,.0f}  → error en las mismas unidades que el gasto')
print(f'R²:    {r2:.4f}  → el modelo explica el {r2*100:.1f}% de la variabilidad del gasto')
print()
print(f'RMSE / std(Y): {rmse/Y.std():.3f}')

### 3.5 Visualización del modelo ajustado

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Modelo de regresión — ŷ = ${b0_g:,.0f} + ${b1_g:,.0f}·X  |  R²={r2:.3f}',
             fontweight='bold')

# 1. Recta de regresión
axes[0].scatter(df_clean['tiempo_navegacion'], Y, color='#2E75B6',
                alpha=0.5, s=30, label='Datos reales')
x_range = np.linspace(df_clean['tiempo_navegacion'].min(),
                      df_clean['tiempo_navegacion'].max(), 100)
axes[0].plot(x_range, b0_g + b1_g * x_range, color='#ED7D31',
             linewidth=2.5, label='Recta OLS')
axes[0].set_title('Recta de regresión')
axes[0].set_xlabel('Tiempo de navegación (min)')
axes[0].set_ylabel('Gasto ($)')
axes[0].legend()

# 2. Valores reales vs predichos
axes[1].scatter(Y, pred, color='#70AD47', alpha=0.6, s=30)
lim = max(Y.max(), pred.max())
axes[1].plot([0, lim], [0, lim], color='red', linestyle='--', linewidth=2,
             label='Predicción perfecta')
axes[1].set_title('Valores reales vs predichos')
axes[1].set_xlabel('Y real')
axes[1].set_ylabel('ŷ predicho')
axes[1].legend(fontsize=8)

# 3. Residuos vs predichos
residuos = Y - pred
axes[2].scatter(pred, residuos, color='#7030A0', alpha=0.6, s=30)
axes[2].axhline(0, color='red', linestyle='--', linewidth=2)
axes[2].axhline(rmse,  color='green', linestyle=':', linewidth=1.5, label=f'+RMSE={rmse:,.0f}')
axes[2].axhline(-rmse, color='green', linestyle=':', linewidth=1.5, label=f'-RMSE')
axes[2].set_title('Gráfico de residuos')
axes[2].set_xlabel('ŷ predicho')
axes[2].set_ylabel('Residuo (Y - ŷ)')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

### ✏️ Preguntas de reflexión actividad guiada:

In [ ]:
# ✏️ ¿Es el RMSE alto o bajo en relación al rango de gastos?
print(f'Rango de gasto: ${Y.min():,.0f} — ${Y.max():,.0f}')
print(f'RMSE:            ${rmse:,.0f}')
r_rmse_contexto = ""

# ✏️ ¿El modelo explica bien la variabilidad del gasto? ¿Qué dice el R²?
r_r2_interp = ""

# ✏️ ¿Qué pasaría si intentáramos predecir el gasto de alguien que navega 60 minutos?
pred_60 = b0_g + b1_g * 60
print(f'Predicción para 60 min: ${pred_60:,.0f}')
r_extrapolacion = ""

print(f'\nRMSE en contexto:  {r_rmse_contexto}')
print(f'R² interpretación: {r_r2_interp}')
print(f'Extrapolación 60min: {r_extrapolacion}')

---
## PARTE 4 — Actividad autónoma: modelo individual con nueva muestra

### 4.1 Cargar y explorar `regresion_autonomo.csv`

In [ ]:
# Código exacto de la presentación
df2 = pd.read_csv('regresion_autonomo.csv')
print(df2.describe().round(2))

In [ ]:
print(df2.head(8))
print()
print('=== Columnas adicionales ===')
print(df2.dtypes)

### 4.2 Ajustar el modelo

In [ ]:
# Código exacto de la presentación
X2 = sm.add_constant(df2['tiempo_navegacion'])
Y2 = df2['gasto']
modelo2 = sm.OLS(Y2, X2).fit()
print(modelo2.summary())

### 4.3 Calcular métricas y comparar con el modelo guiado

In [ ]:
# Código exacto de la presentación
pred2 = modelo2.predict(X2)
mae2  = np.mean(np.abs(Y2 - pred2))
rmse2 = np.sqrt(np.mean((Y2 - pred2) ** 2))
r2_2  = modelo2.rsquared

print('=== Métricas del modelo autónomo ===')
print(f'MAE:  ${mae2:,.0f}')
print(f'RMSE: ${rmse2:,.0f}')
print(f'R²:   {r2_2:.4f}')

# Comparación directa
print()
print('=== Comparación: Modelo guiado vs Autónomo ===')
comp = pd.DataFrame({
    'Modelo guiado':   [b0_g, b1_g, mae,  rmse,  r2],
    'Modelo autónomo': [modelo2.params['const'],
                        modelo2.params['tiempo_navegacion'],
                        mae2, rmse2, r2_2]
}, index=['β₀ (intercepto)', 'β₁ (pendiente)', 'MAE', 'RMSE', 'R²'])
print(comp.round(2))

In [ ]:
# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Comparación de modelos — Guiado vs Autónomo', fontweight='bold')

b0_2 = modelo2.params['const']
b1_2 = modelo2.params['tiempo_navegacion']

for ax, df_p, Y_p, b0_p, b1_p, r2_p, titulo, color in zip(
    axes,
    [df_clean, df2],
    [Y, Y2],
    [b0_g, b0_2],
    [b1_g, b1_2],
    [r2, r2_2],
    ['Modelo guiado (navegacion_clientes)', 'Modelo autónomo (regresion_autonomo)'],
    ['#2E75B6', '#70AD47']
):
    ax.scatter(df_p['tiempo_navegacion'], Y_p, color=color, alpha=0.5, s=30)
    x_r = np.linspace(df_p['tiempo_navegacion'].min(),
                      df_p['tiempo_navegacion'].max(), 100)
    ax.plot(x_r, b0_p + b1_p * x_r, color='#ED7D31', linewidth=2.5)
    ax.set_title(f'{titulo}\nŷ=${b0_p:,.0f}+${b1_p:,.0f}·X  |  R²={r2_p:.3f}',
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('Tiempo de navegación (min)')
    ax.set_ylabel('Gasto ($)')

plt.tight_layout()
plt.show()

### ✏️ Preguntas de reflexión actividad autónoma:

In [ ]:
# ✏️ 1. ¿Qué tan preciso es el modelo con esta nueva muestra?
c1 = ""

# ✏️ 2. ¿Qué diferencias observas respecto a los coeficientes del modelo anterior?
c2 = ""

# ✏️ 3. ¿Cuál de las métricas te parece más útil para comunicar el desempeño a gerencia?
c3 = ""

# ✏️ 4. ¿La relación tiempo→gasto se mantiene estable en ambas muestras?
c4 = ""

# ✏️ 5. ¿Qué variable adicional del dataset autónomo podrías incluir para mejorar el modelo?
c5 = ""

print('--- CONCLUSIONES ACTIVIDAD AUTÓNOMA ---')
for i, c in enumerate([c1, c2, c3, c4, c5], 1):
    print(f'{i}. {c}')

---
## 📋 Resumen de funciones y buenas prácticas

| Paso | Función | Descripción |
|------|---------|-------------|
| Preparar X | `sm.add_constant(df['X'])` | Agrega columna de intercepto |
| Ajustar modelo | `sm.OLS(Y, X).fit()` | Mínimos cuadrados ordinarios |
| Ver resultados | `modelo.summary()` | Resumen completo del modelo |
| Coeficientes | `modelo.params` | β₀ y β₁ |
| P-values | `modelo.pvalues` | Significancia estadística |
| Predicciones | `modelo.predict(X)` | Valores ŷ |
| R² | `modelo.rsquared` | Capacidad explicativa |
| MAE | `np.mean(np.abs(Y - pred))` | Error absoluto medio |
| RMSE | `np.sqrt(np.mean((Y - pred)**2))` | Error cuadrático medio raíz |

**Interpretación de métricas:**

| Métrica | Unidades | Penaliza outliers | Cuándo usar |
|---------|----------|-----------------|-------------|
| MAE | Mismas que Y | No | Comunicar error a no técnicos |
| MSE | Unidades² | Sí | Optimización del modelo |
| RMSE | Mismas que Y | Sí | Comparar con std(Y) |
| R² | Sin unidades | — | Capacidad explicativa |

> 💡 **Regla práctica RMSE:** Compara `RMSE / std(Y)`. Si es < 0.3, el modelo es bueno; si es > 0.6, el modelo no es muy informativo.

> 💡 **Siempre visualiza primero:** Un scatterplot antes de ajustar el modelo permite detectar relaciones no lineales, outliers o subgrupos que harían inválido el modelo lineal.